In [ ]:
from datetime import date
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 5)

df = catalog.load('raw/openaire/researchproduct_dev#parquet')

In [ ]:
def _pick_load_dt(df: pd.DataFrame):
    # Si hay una sola fecha en el batch, usala; si hay varias, quedate con la más reciente;
    # si no hay, hoy.
    if 'load_datetime' not in df.columns or df['_load_datetime'].isna().all():
        return date.today()
    vals = df['_load_datetime'].dropna()
    if vals.nunique() == 1:
        return vals.iloc[0]
    return pd.to_datetime(vals).max().date()

In [ ]:
df_research_instances = df[['id','instances']].explode('instances').reset_index(drop=True)

In [ ]:
df_research_instances

In [ ]:
df_research_instances

In [ ]:
df_instances = pd.json_normalize(df_research_instances['instances'])
df_research_instances = pd.concat([df_research_instances['id'], df_instances], axis=1)

In [ ]:
df_research_instances

In [ ]:
df_research_instances = df_research_instances.explode('pids').reset_index(drop=True)

In [ ]:
df_research_instances

In [ ]:
df_research_instances = df_research_instances.explode('urls').reset_index(drop=True)

In [ ]:
df_research_instances

In [ ]:
df_pids = pd.json_normalize(df_research_instances['pids'])
df_research_instances = df_research_instances.drop(columns=['pids'])
df_research_instances = pd.concat([df_research_instances, df_pids], axis=1)

In [ ]:
df_research_instances

In [ ]:
df_research_alternateidentifiers = df_research_instances[['id','alternateIdentifiers']].dropna().explode('alternateIdentifiers').reset_index(drop=True)
df_alternateidentifiers = pd.json_normalize(df_research_alternateidentifiers['alternateIdentifiers'])
df_research_alternateidentifiers = pd.concat([df_research_alternateidentifiers['id'], df_alternateidentifiers], axis=1)

In [ ]:
df_research_alternateidentifiers

In [ ]:
df_research_instances.drop(columns=['alternateIdentifiers'], inplace=True)

In [ ]:
df_research_instances

## Paso 1: Convierto tipos y selecciono columnas con cardinalidad 1 con respecto a cada research product
+ info en https://graph.openaire.eu/docs/data-model/entities/research-product

In [ ]:
def openaire_land_researchproduct_instances(df: pd.DataFrame)-> pd.DataFrame:

    load_dt = _pick_load_dt(df)

    df_research_instances = df[['id','instances']].explode('instances').reset_index(drop=True)

    df_instances = pd.json_normalize(df_research_instances['instances'])
    df_research_instances = pd.concat([df_research_instances['id'], df_instances], axis=1)

    df_research_instances = df_research_instances.explode('pids').reset_index(drop=True)

    df_research_instances = df_research_instances.explode('urls').reset_index(drop=True)

    df_pids = pd.json_normalize(df_research_instances['pids'])
    df_research_instances = df_research_instances.drop(columns=['pids'])

    df_research_instances = pd.concat([df_research_instances, df_pids], axis=1)

    df_research_alternateidentifiers = df_research_instances[['id','alternateIdentifiers']].dropna().explode('alternateIdentifiers').reset_index(drop=True)
    df_alternateidentifiers = pd.json_normalize(df_research_alternateidentifiers['alternateIdentifiers'])
    df_research_alternateidentifiers = pd.concat([df_research_alternateidentifiers['id'], df_alternateidentifiers], axis=1)

    df_research_instances.drop(columns=['alternateIdentifiers'], inplace=True)

    df_research_instances['_load_datetime'] = date.today()
    df_research_alternateidentifiers['_load_datetime'] = date.today()

    return df_research_instances, df_research_alternateidentifiers


In [ ]:
df_research_instances, df_research_alternateidentifiers = openaire_land_researchproduct_instances(df)

In [ ]:
df_research_instances

In [ ]:
df_research_alternateidentifiers